Aprovisionamiento, Inyección de I/O y Cerrojo de Infraestructura (Colab)

Migramos a Colab para sortear el agotamiento de cuota de Colab, manteniendo intacto el paradigma de *Fine-Tuning* nativo de clasificación de secuencias. Este bloque prepara el contenedor efímero para el estándar industrial de Hugging Face.

1. **Inyección de Dependencias:** Se instalan `transformers`, el motor de `datasets` y, críticamente, `accelerate` (dependencia obligatoria para que el orquestador gestione la VRAM en FP16).
2. **Cerrojo y Bóveda:** Se adaptan las rutas a la arquitectura nativa de Colab (`/content/drive/MyDrive/SITOR/data/processed` para ingesta de solo-lectura y `/content/drive/MyDrive/SITOR/models` para persistencia I/O). Se levanta una aserción a bajo nivel sobre PyTorch; si el contenedor arranca en CPU, el código abortará para proteger la ejecución, exigiendo el uso del acelerador.

In [1]:
# CELDA 1: Inyección Nativa de Dependencias
# ATENCIÓN: Reinicia obligatoriamente la sesión de Colab tras la instalación.
!pip install -q transformers accelerate datasets fastparquet scipy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 81.4 MB/s eta 0:00:00


In [2]:
# CELDA 2: I/O Persistente y Cerrojo de Hardware (Colab)
import os
import torch
import warnings
from google.colab import drive

warnings.filterwarnings('default')

print("Montando almacenamiento persistente de Google Drive...")
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR'
DATA_PATH = f'{BASE_PATH}/data/gold/train_set_v7.parquet'

print("Auditoría de hardware gráfico en curso...")
assert torch.cuda.is_available(), "FALLO CRÍTICO: Entorno limitado a CPU. Cambia el tipo de entorno y activa la GPU T4."
gpu_name = torch.cuda.get_device_name(0)
print(f"Cerrojo superado. Acelerador CUDA en línea: {gpu_name}")

Montando almacenamiento persistente de Google Drive...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mounted at /content/drive
Auditoría de hardware gráfico en curso...
Cerrojo superado. Acelerador CUDA en línea: NVIDIA A100-SXM4-80GB


Ingesta de Matriz y Extracción de Metadatos Semánticos (Full-Shot)

Con la infraestructura estabilizada en la capa gráfica de Colab, procedemos a cargar la matriz completa de entrenamiento. Al descartar el sobremuestreo contrastivo, el ecosistema de *Pandas* asimila sin riesgo el 100% de la volumetría de negocio en la memoria de la placa base (RAM).

Este bloque orquesta dos operaciones críticas previas a la instanciación de los tensores:
1. **Casteo Defensivo:** Purga estructural de celdas vacías y casteo imperativo a tipo cadena (`str`). Un único registro nulo (NaN) arrastrado como residuo del archivo Parquet desencadenaría una excepción de compilación irreversible en el motor C++ del tokenizador de Hugging Face.
2. **Generación de Diccionarios de Enrutamiento:** Reinstanciamos el `LabelEncoder` para transformar las 89 clases de texto a un espacio numérico denso (0 a 89). Acto seguido, aislamos los mapeos bidireccionales (`id2label` y `label2id`). Esta persistencia es obligatoria: el constructor nativo de `AutoModelForSequenceClassification` exigirá ambos diccionarios para embeberlos en la configuración del artefacto final (`config.json`), garantizando la traducción autónoma de la inferencia en producción.

In [3]:
# CELDA 3: Ingesta Defensiva y Extracción de Metadatos
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("Ingestando matriz de entrenamiento desde disco persistente...")
df_train = pd.read_parquet(DATA_PATH, engine='fastparquet')

print("Ejecutando purga y casteo defensivo en la serie de texto...")
# Bloqueo algorítmico contra celdas corruptas o nulos generados en la descompresión
df_train['full_text'] = df_train['full_text'].fillna("").astype(str)

print("Codificando la variable objetivo (Label Encoding)...")
le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['target_tripleta'])

# Extracción bidireccional estricta (Requisito fundacional de Hugging Face)
id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in enumerate(le.classes_)}
num_classes = len(le.classes_)

print(f"Volumen asimilado: {len(df_train)} tickets | Clases objetivo extraídas: {num_classes}")

Ingestando matriz de entrenamiento desde disco persistente...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Ejecutando purga y casteo defensivo en la serie de texto...
Codificando la variable objetivo (Label Encoding)...
Volumen asimilado: 20109 tickets | Clases objetivo extraídas: 89


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Motor de Telemetría BPO (Métricas Híbridas y Blindaje de Tipos)

Este módulo evalúa el rendimiento del algoritmo calculando tanto el rigor estadístico clásico como las métricas de negocio puras (Tasa de Automatización bajo el umbral de seguridad del 0.60). Su aislamiento responde al principio innegociable de mantener la paridad evaluativa frente al *Random Forest* (Fase 2).

Para evitar corrupciones de memoria, el motor incluye un sobreajuste estático de validación estricta (actualizada al estándar *Array API* de NumPy 2.0) que aborta la ejecución si detecta tensores anclados a la VRAM, obligando al orquestador maestro a gestionar la desvinculación de *hardware* antes de la evaluación.

In [4]:
# CELDA 4: Motor de Métricas BPO
import torch
import numpy as np
from sklearn.metrics import f1_score, log_loss, cohen_kappa_score

def calcular_metricas_bpo(y_true, y_prob, umbral=0.60):
    """
    Evaluador central de métricas híbridas (Negocio + Data Science).
    Restricción técnica: y_prob DEBE ser un array plano de NumPy.
    """
    # Trampa de tipos estricta (Compatible con la actualización NumPy 2.0)
    if torch.is_tensor(y_prob):
        raise TypeError("Violación de aislamiento: El motor BPO no acepta tensores de VRAM. Transfiere a NumPy (.cpu().numpy()) antes de inyectar.")

    y_pred = np.argmax(y_prob, axis=1)
    confianzas = np.max(y_prob, axis=1)

    # Segmentación táctica en base a la tolerancia operativa de negocio
    automatizados_mask = confianzas >= umbral

    total_tickets = len(y_true)
    tickets_automatizados = np.sum(automatizados_mask)
    tasa_automatizacion = tickets_automatizados / total_tickets if total_tickets > 0 else 0.0

    # Precisión matemática calculada exclusivamente sobre la capa de automatización
    if tickets_automatizados > 0:
        precision_condicionada = np.mean(y_true[automatizados_mask] == y_pred[automatizados_mask])
    else:
        precision_condicionada = 0.0

    # Telemetría de validación científica (Rigor estadístico)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    # Blindaje dimensional contra omisión de clases en el particionado
    entropia = log_loss(y_true, y_prob, labels=range(num_classes))

    return {
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'kappa': kappa,
        'log_loss': entropia,
        'tasa_automatizacion': tasa_automatizacion,
        'precision_condicionada': precision_condicionada
    }

Bucle Maestro K-Fold (Contingencia Industrial: RoBERTa con Penalización Tensional Suavizada)

Tras diagnosticar la limitación computacional del tensor de pesos lineal (hundimiento de la Tasa de Automatización), se re-diseña la Función de Coste Ponderada aplicando un suavizado de raíz cuadrada inversa (`1/sqrt(f)`). Esto mitiga la explosión de gradientes en las clases minoritarias, permitiendo a la red recuperar confianza estadística sobre las colas masivas.

El orquestador ejecuta seis capas defensivas:
1. **Pre-Tokenización Vectorizada:** El *dataset* de Arrow se tokeniza en bloque (truncamiento a 256 *tokens*).
2. **Imposición Geométrica:** Forzado del mapa de colas (`num_classes=90`) sobre el artefacto base (`ignore_mismatched_sizes=True`).
3. **Cálculo Vectorial de Pesos Suavizados:** Se genera un tensor de penalización escalado (`np.sqrt`) para proteger la métrica *F1-Weighted* sin descuidar el *F1-Macro*.
4. **Sobreescritura del Orquestador (Subclassing):** Se puentea la API estándar instanciando `BPOWeightedTrainer`.
5. **Alineación de Hardware:** El tensor de penalización se transfiere a la VRAM dinámicamente (`.to(device)`).
6. **Desvinculación de Hardware:** Extracción a NumPy y aplicación de `Softmax` para inferencia pura.

In [5]:
# CELDA 5: Orquestador K-Fold Nativo (Contingencia RoBERTa con Pesos Penalizados Suavizados)
import os
import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from scipy.special import softmax

folds_unicos = sorted(df_train['fold_id'].unique())
resultados_kfold = []

# Ajuste de persistencia nativa en Colab
CSV_BACKUP_PATH = f"{BASE_PATH}/data/resultados_kfold_roberta.csv"
if os.path.exists(CSV_BACKUP_PATH):
    os.remove(CSV_BACKUP_PATH)

print("Calculando tensor de penalización suavizado (Square Root Class Weights)...")
etiquetas_globales = np.array(df_train['label'])
clases_unicas, frecuencias = np.unique(etiquetas_globales, return_counts=True)

# Suavizado matemático: 1 / sqrt(frecuencia) para comprimir la penalidad y salvar a la clase mayoritaria
pesos_crudos = 1.0 / np.sqrt(frecuencias)

# Normalización para mantener la magnitud de los gradientes alineada con el Learning Rate
pesos_numpy = pesos_crudos * (len(clases_unicas) / np.sum(pesos_crudos))

# Instanciamos el tensor (se enviará a CUDA dinámicamente en el Trainer para evitar fallos físicos)
tensor_pesos = torch.tensor(pesos_numpy, dtype=torch.float32)

print("Iniciando transición de memoria a Apache Arrow...")
dataset_global = Dataset.from_pandas(df_train[['full_text', 'label', 'fold_id']])
del df_train
gc.collect()

print("Ejecutando tokenización masiva en frío (Truncamiento a 256 tokens)...")
modelo_id = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)

def tokenizar_lote(batch):
    return tokenizer(batch["full_text"], padding="max_length", truncation=True, max_length=256)

dataset_tokenizado = dataset_global.map(tokenizar_lote, batched=True, batch_size=1000)
dataset_tokenizado = dataset_tokenizado.remove_columns(["full_text"])
del dataset_global
gc.collect()

# Sobreescritura del orquestador estándar para inyectar la función de coste penalizada
class BPOWeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Aserción y alineación dinámica de hardware para evitar colisiones RAM/VRAM
        if self.class_weights.device != logits.device:
            self.class_weights = self.class_weights.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

print("\nInicializando motor K-Fold sobre GPU de Colab (RoBERTa FP16 + SqRt Class Weights)...")

for fold in folds_unicos:
    print(f"\n--- Compilando Fold {fold}/{len(folds_unicos)} ---")

    train_dataset = dataset_tokenizado.filter(lambda x: x['fold_id'] != fold)
    val_dataset = dataset_tokenizado.filter(lambda x: x['fold_id'] == fold)

    model = AutoModelForSequenceClassification.from_pretrained(
        modelo_id,
        num_labels=num_classes,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )

    args = TrainingArguments(
        output_dir="/content/tmp_trainer",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=4,
        fp16=True,
        num_train_epochs=10,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        learning_rate=1e-4,
        warmup_steps=300,
        weight_decay=0.01,
        max_grad_norm=1.0
    )

    trainer = BPOWeightedTrainer(
        class_weights=tensor_pesos,
        model=model,
        args=args,
        train_dataset=train_dataset
    )

    print("Inyectando tensores, matriz de pesos y actualizando pesos en CUDA...")
    trainer.train()

    print("Transfiriendo partición de validación e infiriendo Logits crudos...")
    salida_prediccion = trainer.predict(val_dataset)
    logits = salida_prediccion.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    if torch.is_tensor(logits):
        logits = logits.cpu().numpy()

    print("Normalizando Logits mediante Softmax...")
    y_prob = softmax(logits, axis=1)
    y_true = np.array(val_dataset['label'])

    metricas_fold = calcular_metricas_bpo(y_true, y_prob)
    metricas_fold['fold'] = fold
    resultados_kfold.append(metricas_fold)

    print(f"Fold {fold} estabilizado. Tasa Automatización: {metricas_fold['tasa_automatizacion']:.2%}")

    df_fold = pd.DataFrame([metricas_fold])
    insertar_cabecera = not os.path.exists(CSV_BACKUP_PATH)
    df_fold.to_csv(CSV_BACKUP_PATH, mode='a', header=insertar_cabecera, index=False)

    del train_dataset
    del val_dataset
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nBake-Off estadístico finalizado con éxito. Matriz consolidada en disco.")
print("\nVolcado automático de métricas (CSV) por pantalla para auditoría inmediata:")
import pandas as pd
try:
    df_final = pd.read_csv(CSV_BACKUP_PATH)
    print(df_final.to_string())
except Exception as e:
    print(f"No se pudo imprimir el CSV: {e}")

Calculando tensor de penalización suavizado (Square Root Class Weights)...
Iniciando transición de memoria a Apache Arrow...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Ejecutando tokenización masiva en frío (Truncamiento a 256 tokens)...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Transfiriendo partición de validación e infiriendo Logits crudos...


Normalizando Logits mediante Softmax...
Fold 4 estabilizado. Tasa Automatización: 4.75%


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Bake-Off estadístico finalizado con éxito. Matriz consolidada en disco.

Volcado automático de métricas (CSV) por pantalla para auditoría inmediata:
   f1_macro  f1_weighted     kappa  log_loss  tasa_automatizacion  precision_condicionada  fold
0  0.250829     0.263875  0.252665  2.687139             0.029836                0.816667     0
1  0.300173     0.294864  0.285192  2.620567             0.063899                0.688716     1
2  0.280747     0.273478  0.263404  2.674163             0.045002                0.657459     2
3  0.272070     0.270249  0.261297  2.631464             0.042019                0.710059     3
4  0.273246     0.272913  0.263825  2.648665             0.047501                0.759162     4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Entrenamiento Maestro y Auditoría de Producción (Hold-Out Ciego)

Superada la validación cruzada y bloqueados los hiperparámetros de la arquitectura, procedemos a la compilación del artefacto definitivo para el Trabajo de Fin de Máster.

Este orquestador final despliega las siguientes fases críticas:
1. **Ingesta Full-Shot:** Entrenamiento ininterrumpido sobre el 100% de la matriz de entrenamiento (`train_set_v7.parquet`), maximizando la exposición geométrica del modelo a la taxonomía purgada de 89 clases.
2. **Violación de Cuarentena (Test Set):** Por primera y única vez en todo el ciclo de vida del proyecto, se desencripta el `test_set_v7.parquet` (Hold-Out del 20%).
3. **Telemetría Pura:** La red infiere probabilísticamente los tickets ciegos y se extrae el F1-Macro, F1-Weighted, Tasa de Automatización y Precisión Condicionada.
4. **Cierre de I/O:** Persistencia automática del modelo y el tokenizador en la bóveda `/content/drive/MyDrive/SITOR/models/produccion_roberta`.

In [6]:
# CELDA 6: Entrenamiento Maestro y Auditoría Hold-Out Ciego
import os
import gc
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from scipy.special import softmax

print("\n=======================================================")
print("FASE FINAL: ENTRENAMIENTO MAESTRO Y AUDITORÍA HOLD-OUT")
print("=======================================================\n")

print("Ingestando matriz de entrenamiento Full-Shot...")
df_train_full = pd.read_parquet(DATA_PATH, engine='fastparquet')
df_train_full['full_text'] = df_train_full['full_text'].fillna("").astype(str)

print("Generando variables objetivo y tensor suavizado maestro...")
df_train_full['label'] = le.transform(df_train_full['target_tripleta'])

clases_unicas_full, frecuencias_full = np.unique(df_train_full['label'], return_counts=True)
pesos_crudos_full = 1.0 / np.sqrt(frecuencias_full)
pesos_numpy_full = pesos_crudos_full * (len(clases_unicas_full) / np.sum(pesos_crudos_full))
tensor_pesos_maestro = torch.tensor(pesos_numpy_full, dtype=torch.float32)

print("Transicionando a Apache Arrow y tokenizando en frío...")
dataset_train_full = Dataset.from_pandas(df_train_full[['full_text', 'label']])
del df_train_full
gc.collect()

dataset_train_tokenizado = dataset_train_full.map(tokenizar_lote, batched=True, batch_size=1000)
dataset_train_tokenizado = dataset_train_tokenizado.remove_columns(['full_text'])
del dataset_train_full
gc.collect()

print("\n--- Compilando Artefacto de Producción (RoBERTa 125M) ---")
modelo_maestro = AutoModelForSequenceClassification.from_pretrained(
    modelo_id,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

args_maestro = TrainingArguments(
    output_dir="/content/tmp_maestro",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=True,
    num_train_epochs=10,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    learning_rate=1e-4,
    warmup_steps=300,
    weight_decay=0.01,
    max_grad_norm=1.0
)

trainer_maestro = BPOWeightedTrainer(
    class_weights=tensor_pesos_maestro,
    model=modelo_maestro,
    args=args_maestro,
    train_dataset=dataset_train_tokenizado
)

print("Optimizando tensores masivos en CUDA...")
trainer_maestro.train()

print("\n=======================================================")
print("INFERENCIA SOBRE MATRIZ HOLD-OUT CIEGA")
print("=======================================================\n")

# ATENCIÓN: Esta ruta asume que subiste el test_set_v7.parquet a Colab.
# Si el nombre de tu Dataset en Colab es diferente, cambia DATA_PATH_TEST.
DATA_PATH_TEST = f'{BASE_PATH}/data/gold/test_set_v7.parquet'

try:
    print("Ingestando Hold-Out Test Set...")
    df_test = pd.read_parquet(DATA_PATH_TEST, engine='fastparquet')
    df_test['full_text'] = df_test['full_text'].fillna("").astype(str)
    df_test['label'] = le.transform(df_test['target_tripleta'])

    dataset_test = Dataset.from_pandas(df_test[['full_text', 'label']])
    dataset_test_tokenizado = dataset_test.map(tokenizar_lote, batched=True, batch_size=1000)
    dataset_test_tokenizado = dataset_test_tokenizado.remove_columns(['full_text'])

    print("Infiriendo red neuronal sobre matriz ciega...")
    predicciones_test = trainer_maestro.predict(dataset_test_tokenizado)
    logits_test = predicciones_test.predictions

    if isinstance(logits_test, tuple):
        logits_test = logits_test[0]

    if torch.is_tensor(logits_test):
        logits_test = logits_test.cpu().numpy()

    print("Normalizando distribuciones (Softmax)...")
    y_prob_test = softmax(logits_test, axis=1)
    y_true_test = np.array(dataset_test_tokenizado['label'])

    print("Calculando Telemetría Final BPO...\n")
    metricas_produccion = calcular_metricas_bpo(y_true_test, y_prob_test)

    print("--- VEREDICTO DE NEGOCIO ---")
    print(f"F1-Macro:               {metricas_produccion['f1_macro']:.4f}")
    print(f"F1-Weighted:            {metricas_produccion['f1_weighted']:.4f}")
    print(f"Log-Loss:               {metricas_produccion['log_loss']:.4f}")
    print(f"Tasa de Automatización: {metricas_produccion['tasa_automatizacion']:.2%}")
    print(f"Precisión Condicionada: {metricas_produccion['precision_condicionada']:.2%}\n")

    # Inyección de persistencia para el Hold-Out ciego
    df_metricas_produccion = pd.DataFrame([metricas_produccion])
    CSV_PRODUCCION_PATH = f"{BASE_PATH}/data/resultados_holdout_maestro.csv"
    df_metricas_produccion.to_csv(CSV_PRODUCCION_PATH, index=False)
    print(f"Métricas maestras de producción exportadas a: {CSV_PRODUCCION_PATH}\n")

except Exception as e:
    print(f"ERROR CRÍTICO LEYENDO EL TEST SET: {e}\n¡Asegúrate de que el path DATA_PATH_TEST es correcto en Colab!")

print("Persistiendo orquestador y artefactos a disco...")
DIR_PRODUCCION = f"{BASE_PATH}/modelo/produccion_roberta"
trainer_maestro.save_model(DIR_PRODUCCION)
tokenizer.save_pretrained(DIR_PRODUCCION)
print(f"\nArquitectura sellada con éxito en: {DIR_PRODUCCION}")


FASE FINAL: ENTRENAMIENTO MAESTRO Y AUDITORÍA HOLD-OUT

Ingestando matriz de entrenamiento Full-Shot...
Generando variables objetivo y tensor suavizado maestro...
Transicionando a Apache Arrow y tokenizando en frío...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Map:   0%|          | 0/20109 [00:00<?, ? examples/s]


--- Compilando Artefacto de Producción (RoBERTa 125M) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Infiriendo red neuronal sobre matriz ciega...


Normalizando distribuciones (Softmax)...
Calculando Telemetría Final BPO...

--- VEREDICTO DE NEGOCIO ---
F1-Macro:               0.3811
F1-Weighted:            0.3388
Log-Loss:               2.4563
Tasa de Automatización: 13.66%
Precisión Condicionada: 73.51%

Métricas maestras de producción exportadas a: /content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR/data/resultados_holdout_maestro.csv

Persistiendo orquestador y artefactos a disco...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Arquitectura sellada con éxito en: /content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR/modelo/produccion_roberta
